# Auto-Labeling Pipeline — Stage 1: Image-Only, Single Frame, CAM_FRONT

**Modules:**
1. SAM3 segmentation (inline text-prompted, all object classes)
2. SAM3D Body inference (pedestrians)
3. SAM3D Objects inference (all other classes, seeded with SAM3 masks + MoGe pointmap)
4. Orientation estimation (`facing_direction` for pedestrians; ambiguous long axis for objects)
5. OBB fitting (oriented bounding box in ego frame)
6. 3D visualization (k3d)

**Assumptions for stage 1:**
- No LiDAR integration (image-only; MoGe depth is relative/non-metric)
- No ground removal
- Single frame, no temporal context
- Object yaw has 180° ambiguity (pedestrians are fully resolved via skeleton)

## 0. Imports

In [8]:
import sys
import os
import shutil
import tempfile
from pathlib import Path

import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes

import plotly.graph_objects as go

## 1. Config

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Dataset selector  →  change this one variable to switch datasets
# ══════════════════════════════════════════════════════════════════════════════
DATASET = 'nuscenes_mini'           # 'ecp'  or  'nuscenes_mini'

# ── Per-dataset defaults ───────────────────────────────────────────────────────
if DATASET == 'nuscenes_mini':
    DATA_ROOT    = '/media/lleba/ECP_Nuscenes_01/nuScenes_mini'
    NUSCENES_VER = 'v1.0-mini'
    SCENE_IDX    = 3     # mini has 10 scenes (0–9)
    FRAME_IDX    = 20
    # LiDAR aggregation — 32-beam sensor → more sweeps to compensate sparse density
    N_BEFORE = 3
    N_AFTER  = 3
    LIDAR_LINES = 32          # nuScenes: 32-beam Velodyne
    # All cameras available in nuScenes
    ALL_DATASET_CAMERAS = [
        'CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
        'CAM_BACK',  'CAM_BACK_LEFT',  'CAM_BACK_RIGHT',
    ]
elif DATASET == 'ecp':
    DATA_ROOT    = '/media/lleba/ECP_Nuscenes_01/output2/ecp2nuscenes'
    NUSCENES_VER = 'v1.0-trainval'
    SCENE_IDX    = 10    # ECP scene (GT available for 10, 11, 15)
    FRAME_IDX    = 560
    # LiDAR aggregation — 64-beam sensor → single sweep already dense
    N_BEFORE = 2
    N_AFTER  = 2
    LIDAR_LINES = 64          # ECP: 64-beam LiDAR
    # ECP frontal cameras (left-to-right order)
    ALL_DATASET_CAMERAS = [
        'CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
    ]
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}  (must be 'ecp' or 'nuscenes_mini')")

# ── Camera selection ──────────────────────────────────────────────────────────
# CAMERA      : primary / reference camera (used for single-camera visualisations
#               and as the alias target when ALL_CAMERAS=True)
# ALL_CAMERAS : True  → run on every camera in ALL_DATASET_CAMERAS
#               False → run on CAMERA only  (set CAMERA to any single camera you want)
CAMERA      = 'CAM_FRONT_RIGHT'
ALL_CAMERAS = False
CAMERAS     = ALL_DATASET_CAMERAS if ALL_CAMERAS else [CAMERA]

# ── LiDAR aggregation toggle ──────────────────────────────────────────────────
# When True, N_BEFORE + anchor + N_AFTER sweeps are combined after ego-motion
# compensation.  N_BEFORE / N_AFTER are set per-dataset above.
USE_AGGREGATION = True

# ── Model checkpoints ─────────────────────────────────────────────────────────
DEV_ROOT = Path('/home/lleba/Thesis/Development')

from huggingface_hub import hf_hub_download
SAM3_CKPT = hf_hub_download(repo_id='facebook/sam3', filename='sam3.pt')

SAM3D_BODY_REPO = DEV_ROOT / 'Models' / 'SAM3D' / 'sam-3d-body'
SAM3D_BODY_CKPT = SAM3D_BODY_REPO / 'checkpoints' / 'sam-3d-body-dinov3' / 'model.ckpt'
SAM3D_BODY_MHR  = SAM3D_BODY_REPO / 'checkpoints' / 'sam-3d-body-dinov3' / 'assets' / 'mhr_model.pt'

SAM3D_OBJ_REPO  = DEV_ROOT / 'Models' / 'SAM3D' / 'sam-3d-objects'
SAM3D_OBJ_CFG   = SAM3D_OBJ_REPO / 'checkpoints' / 'hf' / 'pipeline.yaml'

CFORMER_ROOT = DEV_ROOT / 'Models' / 'CompletionFormer'
CFORMER_CKPT = CFORMER_ROOT / 'checkpoints' / 'KITTIDC' / 'KITTIDC_L1L2.pt'

# ── SAM3 text prompts per class ───────────────────────────────────────────────
SAM3_PROMPTS = {
    'pedestrian':           'body',
    'car':                  'objects',
    'bicycle':              'objects',
    'motorcycle':           'objects',
    'bus':                  'objects',
    'truck':                'objects',
    'construction vehicle': 'objects',
    'trailer':              'objects',
}

# ── Detection thresholds ──────────────────────────────────────────────────────
SAM3_SCORE_THRESH     = 0.5
MIN_MASK_PX           = 100
BODY_BBOX_THRESH      = 0.8
SAM3_IOU_MERGE_THRESH = 0.4

# ── Pointmap mode ─────────────────────────────────────────────────────────────
#   'baseline'        — dense MoGe relative-depth pointmap (no LiDAR)
#   'o1_lidar'        — sparse metric LiDAR pointmap
#   'o2_moge_affine'  — dense MoGe + global affine fit to LiDAR
#   'o3_local_affine' — dense MoGe + per-object affine fit (HDBSCAN-filtered)
#   'o4_dense_pc'     — CompletionFormer dense metric depth (no MoGe)
POINTMAP_MODE = 'o3_local_affine'

# ── Ego-body exclusion zone ───────────────────────────────────────────────────
USE_EGO_BODY_FILTER = True
EGO_BOX_HALF_X = 4.0
EGO_BOX_HALF_Y = 1.5
EGO_BOX_Z_MIN  = 0.5
EGO_BOX_Z_MAX  = 2.5

# ── Max LiDAR range filter ────────────────────────────────────────────────────
MAX_LIDAR_RANGE_M = 52.0

# ── OBB minimum dimension filter (lenient — only catches obviously broken OBBs) ──
# Pedestrian OBB is camera-space [W, H, D]; only H (vertical) is reliable.
# Object OBBs are ego-space [L, W, H]; L = long axis from PCA.
# Set a key to None or omit it to skip that dimension check.
OBB_MIN_DIMS = {
    'pedestrian':           dict(min_h=0.5),   # H < 0.5 m → ghost/bad depth
    'bicycle':              dict(min_l=0.4),
    'motorcycle':           dict(min_l=0.6),
    'car':                  dict(min_l=1.5),
    'truck':                dict(min_l=2.0),
    'bus':                  dict(min_l=3.0),
    'trailer':              dict(min_l=2.0),
    'construction vehicle': dict(min_l=1.2),
}

USE_OBB_MIN_DIMS_FILTER = True   # set False to skip the size check entirely

# ── BEV: drop non-local ptmap objects ────────────────────────────────────────
# When True, pseudo-label OBBs whose depth was estimated via global_fallback or
# unscaled_fallback are hidden in the BEV plot.  Set False to see everything.
# Only affects the visualisation — re-run just the BEV cell to toggle.
BEV_DROP_NON_LOCAL = False

# ── HDBSCAN — per-class parameters ────────────────────────────────────────────
HDBSCAN_PARAMS = {
    'pedestrian':           dict(min_cluster_size=3, min_samples=1, cluster_eps=0.20),
    'bicycle':              dict(min_cluster_size=3, min_samples=1, cluster_eps=0.20),
    'motorcycle':           dict(min_cluster_size=3, min_samples=1, cluster_eps=0.30),
    'car':                  dict(min_cluster_size=5, min_samples=1, cluster_eps=0.50),
    'truck':                dict(min_cluster_size=5, min_samples=1, cluster_eps=0.60),
    'construction vehicle': dict(min_cluster_size=5, min_samples=1, cluster_eps=0.60),
    'bus':                  dict(min_cluster_size=5, min_samples=1, cluster_eps=0.80),
    'trailer':              dict(min_cluster_size=5, min_samples=1, cluster_eps=0.70),
}

_agg_str = f'aggregated ({N_BEFORE}+1+{N_AFTER} sweeps)' if USE_AGGREGATION else 'single sweep'
print(f'Dataset : {DATASET}  |  scene={SCENE_IDX}  frame={FRAME_IDX}')
print(f'Cameras : {CAMERAS}')
print(f'LiDAR   : {_agg_str}')
print(f'Pointmap: {POINTMAP_MODE!r}')
print(f'Ego-body filter: {"ON" if USE_EGO_BODY_FILTER else "OFF"}  '
      f'(|X|<{EGO_BOX_HALF_X}m, |Y|<{EGO_BOX_HALF_Y}m, Z∈[{EGO_BOX_Z_MIN}, {EGO_BOX_Z_MAX}]m)')
print(f'Max range: {MAX_LIDAR_RANGE_M} m')
print(f'SAM3 checkpoint: {SAM3_CKPT}')
if POINTMAP_MODE == 'o4_dense_pc':
    print(f'CompletionFormer: {CFORMER_CKPT}  exists={CFORMER_CKPT.exists()}')

## 2. Path Setup
Models are loaded sequentially — each section loads its model, runs inference, then unloads before the next model is loaded. This keeps peak GPU memory to one model at a time.

In [ ]:
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_float32_matmul_precision('high')

# Add all repo paths once here so each section can import without repeating sys.path inserts
sys.path.insert(0, str(SAM3D_BODY_REPO))
sys.path.insert(0, str(SAM3D_OBJ_REPO / 'notebook'))
sys.path.insert(0, str(SAM3D_OBJ_REPO))
os.environ['LIDRA_SKIP_INIT'] = 'true'

print('Paths configured. Models will be loaded sequentially in each section.')

## 3. Data Loading
Load the selected frame: image, camera intrinsics, and camera-to-ego extrinsics.

In [ ]:
def load_frame(nusc, scene_idx, frame_idx, camera='CAM_FRONT'):
    """
    Load image, intrinsics, and camera-to-ego extrinsics for one frame.

    frame_idx indexes raw sample_data frames (~12 Hz), NOT keyframes (~2 Hz).
    Use is_key_frame to check whether a given frame has GT annotations.

    Returns
    -------
    img_bgr       : (H, W, 3) uint8 BGR image
    img_rgb       : (H, W, 3) uint8 RGB image
    K             : (3, 3) float64 camera intrinsics
    R_c2e         : (3, 3) float64 rotation camera -> ego
    t_c2e         : (3,)   float64 translation camera -> ego
    img_path      : str    absolute path to image file
    cam_sd_token  : str    nuScenes sample_data token for this camera frame
    """
    scene   = nusc.scene[scene_idx]
    token   = nusc.get('sample', scene['first_sample_token'])['data'][camera]
    for _ in range(frame_idx):
        rec   = nusc.get('sample_data', token)
        token = rec['next']
        assert token, f'frame_idx {frame_idx} exceeds scene length'

    rec      = nusc.get('sample_data', token)
    img_path = nusc.get_sample_data_path(token)
    cal      = nusc.get('calibrated_sensor', rec['calibrated_sensor_token'])

    K     = np.array(cal['camera_intrinsic'], dtype=np.float64)
    R_c2e = Quaternion(cal['rotation']).rotation_matrix.astype(np.float64)
    t_c2e = np.array(cal['translation'], dtype=np.float64)

    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb, img_bgr, K, R_c2e, t_c2e, img_path, token


nusc = NuScenes(version=NUSCENES_VER, dataroot=DATA_ROOT, verbose=False)

# ── Scene / frame metadata ────────────────────────────────────────────────────
_scene_rec  = nusc.scene[SCENE_IDX]
scene_name  = _scene_rec['name']         # e.g. "scene-0061"
n_keyframes = _scene_rec['nbr_samples']  # annotated frames at 2 Hz

# Total raw camera frames in this scene (walk sample_data chain once for CAM_FRONT)
_t_raw = nusc.get('sample', _scene_rec['first_sample_token'])['data'][CAMERA]
_n_raw = 0
while _t_raw:
    _n_raw += 1
    _t_raw  = nusc.get('sample_data', _t_raw)['next']

print(f'Scene index   : {SCENE_IDX} → {scene_name}')
print(f'Raw frames    : {_n_raw}  (camera ~12 Hz, FRAME_IDX range: 0–{_n_raw-1})')
print(f'Keyframes     : {n_keyframes}  (annotated at ~2 Hz, pipeline processes these)')
print()

# ── Load all requested cameras ────────────────────────────────────────────────
cam_data = {}
for _cam in CAMERAS:
    _img_rgb, _img_bgr, _K, _R_c2e, _t_c2e, _img_path, _sd_token = load_frame(
        nusc, SCENE_IDX, FRAME_IDX, _cam
    )
    _H, _W = _img_rgb.shape[:2]
    _fx, _fy, _cx, _cy = _K[0,0], _K[1,1], _K[0,2], _K[1,2]
    cam_data[_cam] = dict(
        img_rgb=_img_rgb, img_bgr=_img_bgr, K=_K,
        R_c2e=_R_c2e, t_c2e=_t_c2e,
        img_path=_img_path, cam_sd_token=_sd_token,
        H=_H, W=_W, fx=_fx, fy=_fy, cx=_cx, cy=_cy,
    )
    print(f'  {_cam}: {_W}x{_H}  {_img_path}')

# ── Ego pose at anchor timestamp (shared across all cameras at a keyframe) ────
_anchor_sd = nusc.get('sample_data', cam_data[CAMERA]['cam_sd_token'])
_anchor_ep = nusc.get('ego_pose', _anchor_sd['ego_pose_token'])
R_e2g_anchor = Quaternion(_anchor_ep['rotation']).rotation_matrix.astype(np.float64)
t_e2g_anchor = np.array(_anchor_ep['translation'], dtype=np.float64)

# ── Is this frame a keyframe? Which keyframe index / nearest keyframes? ───────
def _keyframe_index(nusc, scene_rec, sample_token):
    """Return 0-based keyframe index for a given sample token."""
    _t, _i = scene_rec['first_sample_token'], 0
    while _t and _t != sample_token:
        _t = nusc.get('sample', _t)['next']
        _i += 1
    return _i

_sd_rec = nusc.get('sample_data', cam_data[CAMERA]['cam_sd_token'])
_is_key = _sd_rec['is_key_frame']

if _is_key:
    _kf_idx      = _keyframe_index(nusc, _scene_rec, _sd_rec['sample_token'])
    _frame_label = f'KEYFRAME {_kf_idx}/{n_keyframes-1} (has GT annotations)'
else:
    # Walk backwards to nearest preceding keyframe
    _t_b, _steps_b, _kf_prev = _sd_rec['prev'], 0, None
    while _t_b:
        _steps_b += 1
        _sd_b = nusc.get('sample_data', _t_b)
        if _sd_b['is_key_frame']:
            _kf_prev = _keyframe_index(nusc, _scene_rec, _sd_b['sample_token'])
            break
        _t_b = _sd_b['prev']

    # Walk forwards to nearest following keyframe
    _t_f, _steps_f, _kf_next = _sd_rec['next'], 0, None
    while _t_f:
        _steps_f += 1
        _sd_f = nusc.get('sample_data', _t_f)
        if _sd_f['is_key_frame']:
            _kf_next = _keyframe_index(nusc, _scene_rec, _sd_f['sample_token'])
            break
        _t_f = _sd_f['next']

    _parts = []
    if _kf_prev is not None:
        _parts.append(f'KF {_kf_prev}/{n_keyframes-1} ({_steps_b} back)')
    if _kf_next is not None:
        _parts.append(f'KF {_kf_next}/{n_keyframes-1} ({_steps_f} ahead)')
    _frame_label = f'non-keyframe — nearest: {" | ".join(_parts)}'

print(f'\nFrame {FRAME_IDX}: {_frame_label}')

# ── Primary camera aliases (used by single-camera downstream cells) ───────────
img_rgb, img_bgr  = cam_data[CAMERA]['img_rgb'], cam_data[CAMERA]['img_bgr']
K                 = cam_data[CAMERA]['K']
R_c2e, t_c2e     = cam_data[CAMERA]['R_c2e'], cam_data[CAMERA]['t_c2e']
img_path          = cam_data[CAMERA]['img_path']
cam_sd_token      = cam_data[CAMERA]['cam_sd_token']
H, W              = cam_data[CAMERA]['H'], cam_data[CAMERA]['W']
fx, fy, cx, cy    = cam_data[CAMERA]['fx'], cam_data[CAMERA]['fy'], cam_data[CAMERA]['cx'], cam_data[CAMERA]['cy']

# ── Show all cameras ──────────────────────────────────────────────────────────
n_cams = len(CAMERAS)
fig, axes = plt.subplots(1, n_cams, figsize=(14 * n_cams / max(n_cams, 1), 4))
if n_cams == 1:
    axes = [axes]
for ax, _cam in zip(axes, CAMERAS):
    ax.imshow(cam_data[_cam]['img_rgb'])
    ax.axis('off')
    ax.set_title(_cam, fontsize=8)
plt.suptitle(f'Scene {SCENE_IDX} ({scene_name}), raw frame {FRAME_IDX} — {_frame_label}', y=1.01)
plt.tight_layout()
plt.show()

## 3.5 LiDAR Loading and Camera Projection
Load the timestamp-synced LIDAR_TOP sweep and project visible points into the camera frame.
Produces `pts_ego_vis`, `pts_cam_vis`, `u_vis`, `v_vis`, `Z_vis` used downstream by O1.

In [ ]:
# ── Helper: walk LiDAR linked list, skip ECP duplicate file entries ───────────
def _walk_lidar_tokens(nusc, anchor_token, n_before, n_after):
    anchor_path = nusc.get_sample_data_path(anchor_token)
    seen_paths  = {anchor_path}

    backward = []
    tok = nusc.get('sample_data', anchor_token)['prev']
    while tok and len(backward) < n_before:
        path = nusc.get_sample_data_path(tok)
        if path not in seen_paths:
            backward.append(tok)
            seen_paths.add(path)
        tok = nusc.get('sample_data', tok)['prev']

    tokens = []
    for i, t in enumerate(reversed(backward)):
        tokens.append((-(len(backward) - i), t))
    tokens.append((0, anchor_token))

    count = 0
    tok   = nusc.get('sample_data', anchor_token)['next']
    while tok and count < n_after:
        path = nusc.get_sample_data_path(tok)
        if path not in seen_paths:
            count += 1
            tokens.append((count, tok))
            seen_paths.add(path)
        tok = nusc.get('sample_data', tok)['next']

    return tokens   # (rel_idx, token), chronological


def _load_sweep_ego(nusc, tok, R_e2g_anchor, t_e2g_anchor):
    """Load one sweep, transform to anchor ego frame, return (N,3) float64."""
    sd       = nusc.get('sample_data', tok)
    lid_cal  = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
    ego_pose = nusc.get('ego_pose', sd['ego_pose_token'])
    lid_path = nusc.get_sample_data_path(tok)

    R_l2e    = Quaternion(lid_cal['rotation']).rotation_matrix.astype(np.float64)
    t_l2e    = np.array(lid_cal['translation'], dtype=np.float64)
    R_e2g_sw = Quaternion(ego_pose['rotation']).rotation_matrix.astype(np.float64)
    t_e2g_sw = np.array(ego_pose['translation'], dtype=np.float64)

    pts_raw     = np.fromfile(lid_path, dtype=np.float32).reshape(-1, 5)[:, :3].astype(np.float64)
    pts_ego_sw  = (R_l2e @ pts_raw.T).T + t_l2e
    pts_global  = (R_e2g_sw @ pts_ego_sw.T).T + t_e2g_sw
    pts_ego_anc = (R_e2g_anchor.T @ (pts_global - t_e2g_anchor).T).T
    return pts_ego_anc, Path(lid_path).name


# ── Find anchor LiDAR sweep ───────────────────────────────────────────────────
# Uses the primary camera's sample_data token; all cameras at a keyframe share
# the same LIDAR_TOP sample, so the choice of camera here doesn't matter.
cam_sd = nusc.get('sample_data', cam_sd_token)
cam_ts = cam_sd['timestamp']

if cam_sd['is_key_frame']:
    _sample     = nusc.get('sample', cam_sd['sample_token'])
    lidar_token = _sample['data']['LIDAR_TOP']
else:
    lidar_token = min(
        (sd for sd in nusc.sample_data if sd['channel'] == 'LIDAR_TOP'),
        key=lambda sd: abs(sd['timestamp'] - cam_ts),
    )['token']

# ── Load: single sweep or aggregated ─────────────────────────────────────────
if not USE_AGGREGATION:
    pts_ego, _fname = _load_sweep_ego(nusc, lidar_token, R_e2g_anchor, t_e2g_anchor)
    _lidar_sd = nusc.get('sample_data', lidar_token)
    _dt_ms    = abs(cam_ts - _lidar_sd['timestamp']) / 1_000.0
    print(f'LiDAR (single sweep): {len(pts_ego):,} pts  dt={_dt_ms:.1f} ms  ({_fname})')
else:
    _sweep_tokens = _walk_lidar_tokens(nusc, lidar_token, N_BEFORE, N_AFTER)
    _all_pts = []
    print(f'LiDAR aggregation: {len(_sweep_tokens)} unique sweep(s)')
    for _rel_idx, _tok in _sweep_tokens:
        _sweep_pts, _fname = _load_sweep_ego(nusc, _tok, R_e2g_anchor, t_e2g_anchor)
        _sd_info = nusc.get('sample_data', _tok)
        _dt_ms   = (_sd_info['timestamp'] - cam_ts) / 1_000.0
        _all_pts.append(_sweep_pts)
        _tag = '  <- anchor' if _rel_idx == 0 else ''
        print(f'  [{_rel_idx:+d}]  {len(_sweep_pts):6,} pts  dt={_dt_ms:+7.1f} ms  {_fname}{_tag}')
    pts_ego = np.concatenate(_all_pts, axis=0)
    print(f'  Total: {len(pts_ego):,} pts')

# ── Ego-body exclusion zone ───────────────────────────────────────────────────
if USE_EGO_BODY_FILTER:
    _in_ego_box = (
        (np.abs(pts_ego[:, 0]) < EGO_BOX_HALF_X) &
        (np.abs(pts_ego[:, 1]) < EGO_BOX_HALF_Y) &
        (pts_ego[:, 2] > EGO_BOX_Z_MIN) &
        (pts_ego[:, 2] < EGO_BOX_Z_MAX)
    )
    _n_ego  = len(pts_ego)
    pts_ego = pts_ego[~_in_ego_box]
    print(f'  Ego-body filter: removed {_n_ego - len(pts_ego):,} pts → {len(pts_ego):,} remaining')

# ── Max range filter ──────────────────────────────────────────────────────────
_bev_range = np.sqrt(pts_ego[:, 0]**2 + pts_ego[:, 1]**2)
_n_range   = len(pts_ego)
pts_ego    = pts_ego[_bev_range <= MAX_LIDAR_RANGE_M]
print(f'  Range filter ({MAX_LIDAR_RANGE_M} m): removed {_n_range - len(pts_ego):,} pts → {len(pts_ego):,} remaining')

# ── Project LiDAR into each camera ───────────────────────────────────────────
# pts_ego is camera-independent (ego frame); projection differs per camera.
cam_lidar = {}
print(f'\nProjecting {len(pts_ego):,} pts into {len(CAMERAS)} camera(s):')
for _cam, _cd in cam_data.items():
    _K = _cd['K']
    _fx, _fy, _cx, _cy = _K[0,0], _K[1,1], _K[0,2], _K[1,2]
    _H_c, _W_c = _cd['H'], _cd['W']
    _R_c2e, _t_c2e = _cd['R_c2e'], _cd['t_c2e']

    _pts_cam = (_R_c2e.T @ (pts_ego - _t_c2e).T).T
    _Z_cam   = _pts_cam[:, 2]
    _front   = _Z_cam > 0
    _u_proj  = _pts_cam[_front, 0] / _Z_cam[_front] * _fx + _cx
    _v_proj  = _pts_cam[_front, 1] / _Z_cam[_front] * _fy + _cy
    _in_img  = (_u_proj >= 0) & (_u_proj < _W_c) & (_v_proj >= 0) & (_v_proj < _H_c)

    _pts_ego_vis = pts_ego[_front][_in_img].astype(np.float32)
    _u_vis       = _u_proj[_in_img].astype(np.float32)
    _v_vis       = _v_proj[_in_img].astype(np.float32)
    _Z_vis       = _pts_cam[_front][_in_img][:, 2].astype(np.float32)
    _pts_cam_vis = _pts_cam[_front][_in_img].astype(np.float32)

    cam_lidar[_cam] = dict(
        pts_ego_vis=_pts_ego_vis, pts_cam_vis=_pts_cam_vis,
        u_vis=_u_vis, v_vis=_v_vis, Z_vis=_Z_vis,
        H=_H_c, W=_W_c,
    )
    _z_rng = (f'{float(_Z_vis.min()):.1f}–{float(_Z_vis.max()):.1f} m'
              if len(_Z_vis) else 'no points in image')
    print(f'  {_cam}: {len(_pts_ego_vis):,} visible pts  ({_z_rng})')

# ── Primary camera aliases ────────────────────────────────────────────────────
pts_ego_vis = cam_lidar[CAMERA]['pts_ego_vis']
pts_cam_vis = cam_lidar[CAMERA]['pts_cam_vis']
u_vis       = cam_lidar[CAMERA]['u_vis']
v_vis       = cam_lidar[CAMERA]['v_vis']
Z_vis       = cam_lidar[CAMERA]['Z_vis']
_mode_str = f'aggregated ({N_BEFORE}+1+{N_AFTER} sweeps)' if USE_AGGREGATION else 'single sweep'
print(f'\nPrimary camera ({CAMERA}): {len(pts_ego_vis):,} visible pts  [{_mode_str}]')


## 4. SAM3 Segmentation
Run SAM3 inline with a text prompt for each class. A separate session is opened per class (same pattern as `get_sam3_masks_for_nuscenes_trainval.py`).

In [ ]:
from sam3.model_builder import build_sam3_video_predictor

print('Loading SAM3...')
gpus = range(torch.cuda.device_count())
sam3_predictor = build_sam3_video_predictor(gpus_to_use=gpus, checkpoint_path=str(SAM3_CKPT))
print('  SAM3 ready.')


def run_sam3_single_frame(predictor, img_rgb, text_prompt, score_thresh=0.5, min_mask_px=100):
    """
    Run SAM3 with a text prompt on a single frame.

    The image is written to a temp directory as a single-frame 'video',
    then SAM3 propagates over it and the session is closed.

    Parameters
    ----------
    predictor    : SAM3 video predictor instance
    img_rgb      : (H, W, 3) uint8 RGB image
    text_prompt  : str  e.g. 'car', 'pedestrian', 'bicycle and cyclist'
    score_thresh : float  min object probability to keep
    min_mask_px  : int    min mask area to keep

    Returns
    -------
    detections : list of dicts, each with:
        'binary_mask'  : (H, W) bool
        'score'        : float
        'box_xywh'     : (4,) float  normalised [x, y, w, h]
        'prompt'       : str
    """
    tmp_dir = tempfile.mkdtemp(prefix='sam3_frame_')
    try:
        Image.fromarray(img_rgb).save(os.path.join(tmp_dir, '00000.jpg'))
        resp       = predictor.handle_request(dict(type='start_session', resource_path=tmp_dir))
        session_id = resp['session_id']
        predictor.handle_request(dict(
            type='add_prompt', session_id=session_id, frame_index=0, text=text_prompt
        ))
        frame_outputs = {}
        with torch.autocast('cuda', dtype=torch.bfloat16):
            for resp in predictor.handle_stream_request(dict(
                type='propagate_in_video', session_id=session_id
            )):
                frame_outputs[resp['frame_index']] = resp['outputs']
        predictor.handle_request(dict(type='close_session', session_id=session_id))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    out      = frame_outputs.get(0, {})
    obj_ids  = out.get('out_obj_ids', [])
    probs    = np.asarray(out.get('out_probs', []), dtype=np.float32)
    masks    = out.get('out_binary_masks', [])
    boxes    = np.asarray(out.get('out_boxes_xywh', []), dtype=np.float32)

    detections = []
    for i, (oid, prob) in enumerate(zip(obj_ids, probs)):
        if prob < score_thresh:
            continue
        bm = np.asarray(masks[i], dtype=bool)
        if bm.sum() < min_mask_px:
            continue
        detections.append({
            'binary_mask': bm,
            'score':       float(prob),
            'box_xywh':    boxes[i] if len(boxes) > i else None,
            'prompt':      text_prompt,
        })
    return detections


# ── Run SAM3 for all cameras ──────────────────────────────────────────────────
sam3_results_all = {}
for _cam in CAMERAS:
    print(f'\n─── {_cam} ───')
    _img_rgb = cam_data[_cam]['img_rgb']
    _cam_res = {}
    for prompt, pipeline in SAM3_PROMPTS.items():
        print(f'  "{prompt}" ...', end=' ', flush=True)
        dets = run_sam3_single_frame(
            sam3_predictor, _img_rgb, prompt,
            score_thresh=SAM3_SCORE_THRESH, min_mask_px=MIN_MASK_PX
        )
        _cam_res[prompt] = dets
        print(f'{len(dets)} det(s)')
    sam3_results_all[_cam] = _cam_res
    print(f'  Total: {sum(len(v) for v in _cam_res.values())} detection(s)')

# ── Primary camera alias ──────────────────────────────────────────────────────
sam3_results = sam3_results_all[CAMERA]


In [ ]:
# ── Cross-class deduplication ─────────────────────────────────────────────────
CROSS_CLASS_PRIORITY = {
    'pedestrian':           0,
    'car':                  1,
    'bicycle':              2,
    'truck':                3,
    'motorcycle':           4,
    'trailer':              5,
    'bus':                  6,
    'construction vehicle': 7,
}
CROSS_CLASS_IOU_THRESH = 0.5


def dedup_cross_class(results, iou_thresh=CROSS_CLASS_IOU_THRESH, priority=CROSS_CLASS_PRIORITY):
    flat = [(p, d) for p, dets in results.items() for d in dets]
    keep = [True] * len(flat)
    for i in range(len(flat)):
        for j in range(i + 1, len(flat)):
            if not keep[i] or not keep[j]:
                continue
            p_i, d_i = flat[i]
            p_j, d_j = flat[j]
            if p_i == p_j:
                continue
            a, b  = d_i['binary_mask'], d_j['binary_mask']
            inter = np.logical_and(a, b).sum()
            if inter == 0:
                continue
            iou = inter / np.logical_or(a, b).sum()
            if iou < iou_thresh:
                continue
            if priority.get(p_i, 0) >= priority.get(p_j, 0):
                keep[j] = False
            else:
                keep[i] = False

    out = {p: [] for p in results}
    for (p, d), k in zip(flat, keep):
        if k:
            out[p].append(d)

    removed = sum(1 for k in keep if not k)
    print(f'  removed {removed} duplicate(s)  (IoU thresh={iou_thresh})')
    return out


# ── Apply dedup per camera ────────────────────────────────────────────────────
for _cam in CAMERAS:
    print(f'\n─── {_cam} ───')
    sam3_results_all[_cam] = dedup_cross_class(sam3_results_all[_cam])
    _total = sum(len(v) for v in sam3_results_all[_cam].values())
    for p, dets in sam3_results_all[_cam].items():
        if dets:
            print(f'    {p}: {len(dets)}')
    print(f'  → {_total} detection(s)')

# ── Primary camera alias ──────────────────────────────────────────────────────
sam3_results = sam3_results_all[CAMERA]


In [ ]:
# ── Class colour map ──────────────────────────────────────────────────────────
CLASS_COLORS = {
    'pedestrian':           np.array([255,  60,  60], dtype=np.float32),
    'car':                  np.array([ 60, 180,  60], dtype=np.float32),
    'bicycle and cyclist':  np.array([ 60, 120, 255], dtype=np.float32),
    'motorcycle':           np.array([255, 160,   0], dtype=np.float32),
    'bus':                  np.array([180,  60, 255], dtype=np.float32),
    'truck':                np.array([  0, 210, 210], dtype=np.float32),
    'construction vehicle': np.array([255,  80, 180], dtype=np.float32),
    'trailer':              np.array([140, 200,  60], dtype=np.float32),
}
OUTLINE_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

# ── Visualise SAM3 masks for every camera ─────────────────────────────────────
n_cams = len(CAMERAS)
_ncols = min(n_cams, 3)
_nrows = (n_cams + _ncols - 1) // _ncols
fig, axes = plt.subplots(_nrows, _ncols, figsize=(14, 4 * _nrows), squeeze=False)
axes_flat = [ax for row in axes for ax in row]

for _i, _cam in enumerate(CAMERAS):
    ax = axes_flat[_i]
    _img_rgb = cam_data[_cam]['img_rgb']
    _sam3    = sam3_results_all[_cam]
    _H_c, _W_c = _img_rgb.shape[:2]

    overlay = _img_rgb.copy().astype(np.float32)
    outline_layer = np.zeros((_H_c, _W_c), dtype=bool)

    for prompt, dets in _sam3.items():
        col = CLASS_COLORS.get(prompt, np.array([200, 200, 200], dtype=np.float32))
        for d in dets:
            mask = d['binary_mask']
            overlay[mask] = overlay[mask] * 0.45 + col * 0.55

    for prompt, dets in _sam3.items():
        for d in dets:
            mask_u8 = d['binary_mask'].astype(np.uint8) * 255
            dilated = cv2.dilate(mask_u8, OUTLINE_KERNEL)
            outline = (dilated > 0) & ~d['binary_mask']
            outline_layer |= outline
    overlay[outline_layer] = np.array([255, 255, 255], dtype=np.float32)

    ax.imshow(overlay.astype(np.uint8))
    ax.axis('off')
    n_dets = sum(len(v) for v in _sam3.values())
    ax.set_title(f'{_cam}  ({n_dets} det(s))', fontsize=8)

    for prompt, dets in _sam3.items():
        for i, d in enumerate(dets):
            ys, xs = np.where(d['binary_mask'])
            if not len(xs):
                continue
            ax.text(float(xs.mean()), float(ys.mean()), str(i),
                    color='white', fontsize=8, fontweight='bold', ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.15', fc='black', alpha=0.55, ec='none'))

# Legend — total counts across ALL cameras
_class_totals = {}
for _cam in CAMERAS:
    for prompt, dets in sam3_results_all[_cam].items():
        _class_totals[prompt] = _class_totals.get(prompt, 0) + len(dets)
legend_patches = [
    patches.Patch(
        color=CLASS_COLORS.get(prompt, np.array([200, 200, 200], dtype=np.float32)) / 255,
        label=f'{prompt} ({count})',
    )
    for prompt, count in _class_totals.items() if count > 0
]
# Hide unused axes
for ax in axes_flat[n_cams:]:
    ax.axis('off')

if legend_patches and n_cams > 0:
    axes_flat[n_cams - 1].legend(handles=legend_patches, loc='upper right', fontsize=7,
                                  bbox_to_anchor=(1.0, 1.0))

plt.suptitle('SAM3 detections — all cameras (after cross-class dedup)', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Unload SAM3 before loading the next model
del sam3_predictor
gc.collect()
torch.cuda.empty_cache()
print(f'SAM3 unloaded. GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')

## 5. SAM3D Body — Pedestrian Meshes
Uses the internal ViTDet detector supplemented with SAM3 pedestrian detections.

In [ ]:
from sam_3d_body import load_sam_3d_body, SAM3DBodyEstimator
from tools.build_detector import HumanDetector

# SAM3 loading leaves torch.autocast globally enabled (BF16).
# Explicitly disable it before loading SAM3D Body so model init and inference
# are not affected. process_one_image is also wrapped below for safety.
print(f'Autocast state inherited from SAM3: {torch.is_autocast_enabled()}')
torch.autocast('cuda', enabled=False).__enter__()
print(f'Autocast disabled: {torch.is_autocast_enabled()}')

print('Loading SAM3D Body...')
_body_model, _body_cfg = load_sam_3d_body(
    str(SAM3D_BODY_CKPT), device=device, mhr_path=str(SAM3D_BODY_MHR)
)
body_estimator = SAM3DBodyEstimator(
    sam_3d_body_model=_body_model,
    model_cfg=_body_cfg,
    human_detector=HumanDetector(name='vitdet', device=device),
    human_segmentor=None,
    fov_estimator=None,
)
print('  SAM3D Body ready.')


def merge_with_vitdet(sam3_boxes, vitdet_boxes, iou_thresh=0.4):
    """
    Return SAM3 boxes plus any ViTDet box not already covered by a SAM3 box.

    SAM3 boxes take priority; ViTDet only fills gaps (IoU < iou_thresh with all SAM3 boxes).
    """
    if len(vitdet_boxes) == 0:
        return sam3_boxes
    if len(sam3_boxes) == 0:
        return vitdet_boxes
    ix1   = np.maximum(vitdet_boxes[:, 0:1], sam3_boxes[:, 0])
    iy1   = np.maximum(vitdet_boxes[:, 1:2], sam3_boxes[:, 1])
    ix2   = np.minimum(vitdet_boxes[:, 2:3], sam3_boxes[:, 2])
    iy2   = np.minimum(vitdet_boxes[:, 3:4], sam3_boxes[:, 3])
    inter = np.maximum(0.0, ix2 - ix1) * np.maximum(0.0, iy2 - iy1)
    area_v = ((vitdet_boxes[:, 2] - vitdet_boxes[:, 0]) *
              (vitdet_boxes[:, 3] - vitdet_boxes[:, 1]))[:, None]
    area_s = ((sam3_boxes[:, 2] - sam3_boxes[:, 0]) *
              (sam3_boxes[:, 3] - sam3_boxes[:, 1]))[None, :]
    iou   = inter / (area_v + area_s - inter + 1e-8)
    extra = vitdet_boxes[iou.max(axis=1) < iou_thresh]
    if len(extra) == 0:
        return sam3_boxes
    return np.vstack([sam3_boxes, extra])


def run_sam3d_body(estimator, img_path, img_bgr, K, sam3_ped_dets,
                   bbox_thresh=0.8, iou_merge_thresh=0.4):
    """Run SAM3D Body on one image. Returns list of pedestrian result dicts."""
    K_t = torch.tensor(K, dtype=torch.float32).unsqueeze(0)

    sam3_boxes = []
    sam3_mask_indices = []
    for i, d in enumerate(sam3_ped_dets):
        ys, xs = np.where(d['binary_mask'])
        if len(xs) == 0:
            continue
        sam3_boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])
        sam3_mask_indices.append(i)
    sam3_boxes = (np.array(sam3_boxes, dtype=np.float32)
                  if sam3_boxes else np.empty((0, 4), dtype=np.float32))

    vitdet_boxes = estimator.detector.run_human_detection(
        img_bgr, bbox_thr=bbox_thresh, nms_thr=0.3, default_to_full_image=False
    )
    vitdet_boxes = np.asarray(vitdet_boxes, dtype=np.float32).reshape(-1, 4)
    merged_boxes = merge_with_vitdet(sam3_boxes, vitdet_boxes, iou_thresh=iou_merge_thresh)
    n_vitdet_only = len(merged_boxes) - len(sam3_boxes)
    idx_str = ', '.join(str(i) for i in sam3_mask_indices)
    vitdet_str = f' + {n_vitdet_only} ViTDet-only' if n_vitdet_only > 0 else ''
    print(f'    SAM3: {len(sam3_boxes)}  ViTDet: {len(vitdet_boxes)}  '
          f'merged: {len(merged_boxes)}  (SAM3 mask idx: [{idx_str}]{vitdet_str})')

    if len(merged_boxes) == 0:
        return []

    with torch.autocast('cuda', enabled=False):
        outputs = estimator.process_one_image(
            img_path, bboxes=merged_boxes, bbox_thr=bbox_thresh, use_mask=False, cam_int=K_t,
        )

    faces = estimator.faces
    if hasattr(faces, 'cpu'):
        faces = faces.detach().cpu().numpy()
    faces = faces.astype(np.int32)

    results = []
    for o in outputs:
        verts  = np.asarray(o['pred_vertices'],    dtype=np.float32)
        joints = np.asarray(o['pred_keypoints_3d'], dtype=np.float32)
        cam_t  = np.asarray(o['pred_cam_t'],       dtype=np.float32)
        results.append({
            'vertices':  verts + cam_t[None, :],
            'faces':     faces,
            'joints_3d': joints,
            'cam_t':     cam_t,
        })
    return results


# ── Run SAM3D Body for all cameras ────────────────────────────────────────────
print('\nRunning SAM3D Body (pedestrians)...')
body_results_all = {}
for _cam in CAMERAS:
    print(f'\n─── {_cam} ───')
    _cd  = cam_data[_cam]
    _res = run_sam3d_body(
        body_estimator, _cd['img_path'], _cd['img_bgr'], _cd['K'],
        sam3_results_all[_cam].get('pedestrian', []),
        bbox_thresh=BODY_BBOX_THRESH, iou_merge_thresh=SAM3_IOU_MERGE_THRESH,
    )
    body_results_all[_cam] = _res
    print(f'  {len(_res)} pedestrian(s) reconstructed.')

# ── Primary camera alias ──────────────────────────────────────────────────────
body_results = body_results_all[CAMERA]

# ── Unload ────────────────────────────────────────────────────────────────────
del body_estimator, _body_model
gc.collect()
torch.cuda.empty_cache()
print(f'\nSAM3D Body unloaded. GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')


## 6. Pointmap + SAM3D Objects — All Other Class Meshes
SAM3D Objects is loaded once, used for both pointmap computation and per-object inference, then unloaded.

Pointmap mode is controlled by `POINTMAP_MODE` (set in Section 1):

| Mode | Depth source | Coverage | Metric? | Scope |
|---|---|---|---|---|
| `baseline` | MoGe monocular depth | Dense | No (relative) | — |
| `o1_lidar` | Raw LiDAR projection | Sparse (~1–2%) | Yes | Per frame |
| `o2_moge_affine` | MoGe + global LiDAR affine fit | Dense | Yes | Per frame |
| `o3_local_affine` | MoGe + HDBSCAN-filtered per-object affine fit | Dense | Yes | Per object |


In [ ]:
import hdbscan as _hdbscan
from inference import Inference as SAM3DObjectsInference
from pytorch3d.renderer import look_at_view_transform
from pytorch3d.transforms import Transform3d, quaternion_to_matrix

print('Loading SAM3D Objects...')
obj_inference = SAM3DObjectsInference(str(SAM3D_OBJ_CFG), compile=False)
print('  SAM3D Objects ready.')


# ── Baseline: MoGe relative-depth pointmap ────────────────────────────────────

def compute_moge_pointmap(obj_inference, img_rgb, K):
    H, W = img_rgb.shape[:2]
    rgba = np.concatenate([img_rgb, np.full((H, W, 1), 255, dtype=np.uint8)], axis=-1)
    with obj_inference._pipeline.device:
        ptmap_dict = obj_inference._pipeline.compute_pointmap(rgba)
    ptmap_p3d = ptmap_dict['pointmap'].cpu().permute(1, 2, 0).numpy().astype(np.float32)
    _fx, _fy, _cx, _cy = K[0,0], K[1,1], K[0,2], K[1,2]
    u_grid, v_grid = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
    Z_map = ptmap_p3d[..., 2]
    X_r3  = (u_grid - _cx) * Z_map / _fx
    Y_r3  = (v_grid - _cy) * Z_map / _fy
    ptmap = np.stack([-X_r3, -Y_r3, Z_map], axis=-1).astype(np.float32)
    return ptmap, Z_map


# ── O1: sparse LiDAR pointmap ─────────────────────────────────────────────────

def compute_lidar_pointmap(pts_cam_vis, u_vis, v_vis, Z_vis, K, H, W):
    ptmap = np.full((H, W, 3), np.nan, dtype=np.float32)
    u_int = np.round(u_vis).astype(int)
    v_int = np.round(v_vis).astype(int)
    valid = (u_int >= 0) & (u_int < W) & (v_int >= 0) & (v_int < H)
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    X_cam = (u_vis[valid] - cx) * Z_vis[valid] / fx
    Y_cam = (v_vis[valid] - cy) * Z_vis[valid] / fy
    ptmap[v_int[valid], u_int[valid], 0] = -X_cam
    ptmap[v_int[valid], u_int[valid], 1] = -Y_cam
    ptmap[v_int[valid], u_int[valid], 2] =  Z_vis[valid]
    return ptmap


# ── O2: MoGe + global affine calibration ─────────────────────────────────────

def compute_moge_affine_pointmap(obj_inference, img_rgb, K, u_vis, v_vis, Z_vis,
                                  z_min=0.5, z_max=80.0, min_pts=5):
    H, W = img_rgb.shape[:2]
    _, Z_moge_map = compute_moge_pointmap(obj_inference, img_rgb, K)
    u_int = np.round(u_vis).astype(int).clip(0, W-1)
    v_int = np.round(v_vis).astype(int).clip(0, H-1)
    Z_moge_at_lidar = Z_moge_map[v_int, u_int]
    valid = np.isfinite(Z_moge_at_lidar) & (Z_vis > z_min) & (Z_vis < z_max)
    n_fit = int(valid.sum())
    print(f'  Affine fit: {n_fit} pairs')
    if n_fit < min_pts:
        print('  [WARN] Too few pairs — using unscaled MoGe.')
        a_fit, b_fit = 1.0, 0.0
        Z_metric_map = Z_moge_map.copy()
    else:
        Z_m = Z_moge_at_lidar[valid]; Z_l = Z_vis[valid]
        A = np.stack([Z_m, np.ones_like(Z_m)], axis=1)
        (a_fit, b_fit), _, _, _ = np.linalg.lstsq(A, Z_l, rcond=None)
        Z_pred = a_fit * Z_m + b_fit
        rmse = float(np.sqrt(np.mean((Z_pred - Z_l)**2)))
        r2   = float(1.0 - np.sum((Z_l - Z_pred)**2) / np.sum((Z_l - Z_l.mean())**2))
        print(f'  Z_metric = {a_fit:.4f} x Z_moge + {b_fit:.4f}   R²={r2:.4f}  RMSE={rmse:.3f} m')
        Z_metric_map = np.maximum(a_fit * Z_moge_map + b_fit, 0.1).astype(np.float32)
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    u_grid, v_grid = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
    X_r3 = (u_grid - cx) * Z_metric_map / fx
    Y_r3 = (v_grid - cy) * Z_metric_map / fy
    ptmap = np.stack([-X_r3, -Y_r3, Z_metric_map], axis=-1).astype(np.float32)
    return ptmap, Z_metric_map, a_fit, b_fit


# ── O3: per-object local affine calibration with HDBSCAN ─────────────────────

def _fit_affine(Z_moge_vals, Z_lidar_vals, min_pts=4, z_min=0.5, z_max=80.0):
    valid = np.isfinite(Z_moge_vals) & (Z_lidar_vals > z_min) & (Z_lidar_vals < z_max)
    if valid.sum() < min_pts:
        return None
    Z_m = Z_moge_vals[valid]; Z_l = Z_lidar_vals[valid]
    A = np.stack([Z_m, np.ones_like(Z_m)], axis=1)
    (a, b), _, _, _ = np.linalg.lstsq(A, Z_l, rcond=None)
    return a, b


def _build_ptmap_from_affine(Z_moge_map, K, a, b):
    H, W = Z_moge_map.shape
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]
    Z_metric = np.maximum(a * Z_moge_map + b, 0.1).astype(np.float32)
    u_grid, v_grid = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
    X_r3 = (u_grid - cx) * Z_metric / fx
    Y_r3 = (v_grid - cy) * Z_metric / fy
    return np.stack([-X_r3, -Y_r3, Z_metric], axis=-1).astype(np.float32)


def compute_local_affine_ptmap_for_object(Z_moge_map, K,
                                            pts_ego_vis, u_vis, v_vis, Z_vis,
                                            binary_mask, H, W,
                                            min_cluster_size=3, min_samples=1,
                                            cluster_eps=0.4, min_pts=4,
                                            z_min=0.5, z_max=80.0):
    u_int = np.round(u_vis).astype(int).clip(0, W-1)
    v_int = np.round(v_vis).astype(int).clip(0, H-1)

    in_mask = binary_mask[v_int, u_int]
    pts_inmask  = pts_ego_vis[in_mask]
    Z_inmask    = Z_vis[in_mask]
    u_inmask    = u_vis[in_mask]
    v_inmask    = v_vis[in_mask]
    Z_moge_all  = Z_moge_map[v_int, u_int]

    if len(pts_inmask) < min_cluster_size:
        ab = _fit_affine(Z_moge_all, Z_vis, min_pts=min_pts, z_min=z_min, z_max=z_max)
        if ab is None:
            return _build_ptmap_from_affine(Z_moge_map, K, 1.0, 0.0), 1.0, 0.0, 'unscaled_fallback'
        return _build_ptmap_from_affine(Z_moge_map, K, *ab), ab[0], ab[1], 'global_fallback'

    clusterer = _hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric='euclidean',
        cluster_selection_epsilon=cluster_eps,
    )
    labels = clusterer.fit_predict(pts_inmask)
    unique, counts = np.unique(labels[labels >= 0], return_counts=True)

    if len(unique) == 0:
        ab = _fit_affine(Z_moge_all, Z_vis, min_pts=min_pts, z_min=z_min, z_max=z_max)
        if ab is None:
            return _build_ptmap_from_affine(Z_moge_map, K, 1.0, 0.0), 1.0, 0.0, 'unscaled_fallback'
        return _build_ptmap_from_affine(Z_moge_map, K, *ab), ab[0], ab[1], 'global_fallback'

    dominant = unique[np.argmax(counts)]
    keep = labels == dominant
    u_clean_int = np.round(u_inmask[keep]).astype(int).clip(0, W-1)
    v_clean_int = np.round(v_inmask[keep]).astype(int).clip(0, H-1)
    Z_moge_clean = Z_moge_map[v_clean_int, u_clean_int]
    Z_clean      = Z_inmask[keep]

    ab = _fit_affine(Z_moge_clean, Z_clean, min_pts=min_pts, z_min=z_min, z_max=z_max)
    if ab is None:
        ab = _fit_affine(Z_moge_all, Z_vis, min_pts=min_pts, z_min=z_min, z_max=z_max)
        if ab is None:
            return _build_ptmap_from_affine(Z_moge_map, K, 1.0, 0.0), 1.0, 0.0, 'unscaled_fallback'
        return _build_ptmap_from_affine(Z_moge_map, K, *ab), ab[0], ab[1], 'global_fallback'

    return _build_ptmap_from_affine(Z_moge_map, K, *ab), ab[0], ab[1], 'local'


# ── SAM3D Objects inference ───────────────────────────────────────────────────

def mesh_to_r3(output):
    from sam3d_objects.data.dataset.tdfy.transforms_3d import compose_transform

    def _sq(t):
        while t.dim() > 2: t = t.squeeze(0)
        return t

    quat  = _sq(output['rotation'].detach().cpu().float())
    trans = _sq(output['translation'].detach().cpu().float())
    scale = _sq(output['scale'].detach().cpu().float())
    if quat.dim() == 1:
        quat = quat.unsqueeze(0)

    l2c = compose_transform(scale=scale, rotation=quaternion_to_matrix(quat), translation=trans)
    R_r3_to_p3d, _ = look_at_view_transform(
        eye=torch.tensor([[0., 0., -1.]]), at=torch.tensor([[0., 0., 0.]]),
        up=torch.tensor([[0., -1., 0.]]),
    )
    t_p3d_to_r3 = Transform3d().rotate(R_r3_to_p3d).inverse()
    mesh_res    = output['mesh'][0]
    verts_local = mesh_res.vertices.float().cpu()
    faces       = mesh_res.faces.long().cpu().numpy().astype(np.int32)
    xyz_p3d     = l2c.transform_points(verts_local.unsqueeze(0)).squeeze(0)
    verts_r3    = t_p3d_to_r3.transform_points(xyz_p3d.unsqueeze(0)).squeeze(0).numpy().astype(np.float32)
    return verts_r3, faces


def run_sam3d_objects(inference_model, img_rgb, sam3_results, prompts_config, ptmap_t=None,
                       o3_data=None):
    """Run SAM3D Objects for all non-pedestrian classes."""
    results = []
    for prompt, pipeline in prompts_config.items():
        if pipeline != 'objects':
            continue
        dets = sam3_results.get(prompt, [])
        if not dets:
            continue
        print(f'  "{prompt}": {len(dets)} instance(s)...')
        for mask_idx, d in enumerate(dets):
            print(f'    [mask {mask_idx}] processing...')
            try:
                if o3_data is not None:
                    _hp = o3_data.get('hdbscan_params', {}).get(prompt, {})
                    ptmap_obj, a_obj, b_obj, mode = compute_local_affine_ptmap_for_object(
                        Z_moge_map       = o3_data['Z_moge_map'],
                        K                = o3_data['K'],
                        pts_ego_vis      = o3_data['pts_ego_vis'],
                        u_vis            = o3_data['u_vis'],
                        v_vis            = o3_data['v_vis'],
                        Z_vis            = o3_data['Z_vis'],
                        binary_mask      = d['binary_mask'],
                        H                = o3_data['H'],
                        W                = o3_data['W'],
                        min_cluster_size = _hp.get('min_cluster_size', 3),
                        min_samples      = _hp.get('min_samples', 1),
                        cluster_eps      = _hp.get('cluster_eps', 0.4),
                    )
                    print(f'    [mask {mask_idx}] [{mode}]  Z = {a_obj:.4f}*Z_moge + {b_obj:.4f}')
                    ptmap_t_use = torch.tensor(ptmap_obj, dtype=torch.float32)
                else:
                    ptmap_t_use = ptmap_t

                out = inference_model(img_rgb, d['binary_mask'], seed=42, pointmap=ptmap_t_use)
                verts_r3, faces = mesh_to_r3(out)
                results.append({
                    'vertices':    verts_r3,
                    'faces':       faces,
                    'binary_mask': d['binary_mask'],
                    'score':       d['score'],
                    'prompt':      prompt,
                    'ptmap_mode':  mode if o3_data is not None else 'n/a',
                })
            except Exception as e:
                print(f'    [mask {mask_idx}] [WARN] failed for "{prompt}": {e}')
    return results




# ── O4: CompletionFormer dense metric depth ───────────────────────────────────

def compute_dense_completion_pointmap(cformer_model, img_rgb, K, u_vis, v_vis, Z_vis, H, W):
    """
    Run CompletionFormer on one camera view.
    Inputs:
      cformer_model : loaded CompletionFormer (eval mode, on CUDA)
      img_rgb       : (H,W,3) uint8 RGB
      K             : (3,3) camera intrinsics
      u_vis, v_vis  : projected LiDAR pixel coords (float, already clipped)
      Z_vis         : corresponding metric depths (m)
      H, W          : image dimensions
    Returns:
      ptmap : (H,W,3) float32 pointmap in PyTorch3D convention (-X,-Y,Z)
      dense_depth : (H,W) float32 metric depth map
    """
    import torch
    fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]

    # Build sparse depth map (min per pixel)
    u_int = np.clip(np.round(u_vis).astype(int), 0, W - 1)
    v_int = np.clip(np.round(v_vis).astype(int), 0, H - 1)
    _tmp = np.full((H, W), np.inf, dtype=np.float32)
    np.minimum.at(_tmp, (v_int, u_int), Z_vis.astype(np.float32))
    sparse_depth = np.where(_tmp < np.inf, _tmp, 0.0).astype(np.float32)

    # Prepare tensors
    rgb_t = torch.from_numpy(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).cuda()
    dep_t = torch.from_numpy(sparse_depth).unsqueeze(0).unsqueeze(0).cuda()

    # Inference
    with torch.no_grad():
        out = cformer_model({'rgb': rgb_t, 'dep': dep_t})
    dense_depth = out['pred'].squeeze().cpu().numpy()  # (H,W) metres

    # Back-project to (H,W,3) pointmap in PyTorch3D convention (-X,-Y,Z)
    v_grid, u_grid = np.mgrid[0:H, 0:W]
    Z = dense_depth.astype(np.float64)
    X = (u_grid - cx) / fx * Z
    Y = (v_grid - cy) / fy * Z
    ptmap = np.stack([-X, -Y, Z], axis=-1).astype(np.float32)
    return ptmap, dense_depth

# ── Run SAM3D Objects for all cameras ─────────────────────────────────────────

# ── Load CompletionFormer if needed ──────────────────────────────────────────
_cformer = None
if POINTMAP_MODE == 'o4_dense_pc':
    import argparse
    _cf_src = str(CFORMER_ROOT / 'src')
    _cf_dcn = str(CFORMER_ROOT / 'src' / 'model' / 'deformconv')
    for _p in [_cf_src, _cf_dcn]:
        if _p not in sys.path:
            sys.path.insert(0, _p)
    from model.completionformer import CompletionFormer as _CompletionFormer
    _cf_args = argparse.Namespace(
        data_name='KITTIDC', prop_time=6, prop_kernel=3,
        preserve_input=False, affinity='TGASS', affinity_gamma=0.5,
        conf_prop=True, legacy=False, from_scratch=False,
        max_depth=90.0, num_sample=500, lidar_lines=LIDAR_LINES,
    )
    _cformer = _CompletionFormer(_cf_args).cuda()
    _ckpt = torch.load(str(CFORMER_CKPT), map_location='cuda', weights_only=False)
    _cformer.load_state_dict(_ckpt.get('net', _ckpt), strict=True)
    _cformer.eval()
    print(f'CompletionFormer loaded ({sum(p.numel() for p in _cformer.parameters())/1e6:.1f} M params)')

print(f'\nRunning SAM3D Objects ({POINTMAP_MODE!r}) for {len(CAMERAS)} camera(s)...')
obj_results_all = {}
_primary_viz = {}   # save pointmap viz data for primary camera

for _cam in CAMERAS:
    _cd  = cam_data[_cam]
    _cl  = cam_lidar[_cam]
    _K   = _cd['K']
    _H_c, _W_c = _cl['H'], _cl['W']
    print(f'\n─── {_cam} ───')

    if POINTMAP_MODE == 'o3_local_affine':
        print('  Computing MoGe...')
        _, _Z_moge = compute_moge_pointmap(obj_inference, _cd['img_rgb'], _K)
        _o3_data = dict(
            Z_moge_map=_Z_moge, K=_K,
            pts_ego_vis=_cl['pts_ego_vis'],
            u_vis=_cl['u_vis'], v_vis=_cl['v_vis'], Z_vis=_cl['Z_vis'],
            H=_H_c, W=_W_c, hdbscan_params=HDBSCAN_PARAMS,
        )
        _obj_res = run_sam3d_objects(
            obj_inference, _cd['img_rgb'], sam3_results_all[_cam], SAM3_PROMPTS,
            o3_data=_o3_data
        )
        if _cam == CAMERA:
            _primary_viz = dict(Z_map=_Z_moge, mode='o3')
    elif POINTMAP_MODE == 'o1_lidar':
        _ptmap = compute_lidar_pointmap(
            _cl['pts_cam_vis'], _cl['u_vis'], _cl['v_vis'], _cl['Z_vis'], _K, _H_c, _W_c
        )
        _obj_res = run_sam3d_objects(
            obj_inference, _cd['img_rgb'], sam3_results_all[_cam], SAM3_PROMPTS,
            ptmap_t=torch.tensor(_ptmap, dtype=torch.float32)
        )
        if _cam == CAMERA:
            _primary_viz = dict(Z_map=_ptmap[..., 2], mode='o1')
    elif POINTMAP_MODE == 'o2_moge_affine':
        _ptmap, _Z_map, _a, _b = compute_moge_affine_pointmap(
            obj_inference, _cd['img_rgb'], _K, _cl['u_vis'], _cl['v_vis'], _cl['Z_vis']
        )
        _obj_res = run_sam3d_objects(
            obj_inference, _cd['img_rgb'], sam3_results_all[_cam], SAM3_PROMPTS,
            ptmap_t=torch.tensor(_ptmap, dtype=torch.float32)
        )
        if _cam == CAMERA:
            _primary_viz = dict(Z_map=_Z_map, a=_a, b=_b, mode='o2')
    elif POINTMAP_MODE == 'o4_dense_pc':
        _ptmap, _dense_d = compute_dense_completion_pointmap(
            _cformer, _cd['img_rgb'], _K,
            _cl['u_vis'], _cl['v_vis'], _cl['Z_vis'], _H_c, _W_c,
        )
        _obj_res = run_sam3d_objects(
            obj_inference, _cd['img_rgb'], sam3_results_all[_cam], SAM3_PROMPTS,
            ptmap_t=torch.tensor(_ptmap, dtype=torch.float32)
        )
        if _cam == CAMERA:
            _primary_viz = dict(Z_map=_dense_d, mode='o4')
    else:  # baseline
        _ptmap, _Z_map = compute_moge_pointmap(obj_inference, _cd['img_rgb'], _K)
        _obj_res = run_sam3d_objects(
            obj_inference, _cd['img_rgb'], sam3_results_all[_cam], SAM3_PROMPTS,
            ptmap_t=torch.tensor(_ptmap, dtype=torch.float32)
        )
        if _cam == CAMERA:
            _primary_viz = dict(Z_map=_Z_map, mode='baseline')

    obj_results_all[_cam] = _obj_res
    print(f'  {len(_obj_res)} object mesh(es) reconstructed.')

# ── Primary camera alias ──────────────────────────────────────────────────────
obj_results = obj_results_all[CAMERA]

# ── Unload ────────────────────────────────────────────────────────────────────
del obj_inference
if _cformer is not None:
    del _cformer
    _cformer = None
gc.collect()
torch.cuda.empty_cache()
print(f'\nSAM3D Objects unloaded. GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')

# ── Visualise pointmap for primary camera ─────────────────────────────────────
_Z = _primary_viz.get('Z_map')
if _Z is not None:
    _u_prim = cam_lidar[CAMERA]['u_vis']
    _v_prim = cam_lidar[CAMERA]['v_vis']
    _Z_prim = cam_lidar[CAMERA]['Z_vis']
    fig, ax = plt.subplots(figsize=(14, 5))
    if POINTMAP_MODE == 'o1_lidar':
        ax.imshow(cam_data[CAMERA]['img_rgb'])
        sc = ax.scatter(_u_prim, _v_prim, c=_Z_prim, cmap='plasma', s=1, vmin=0, vmax=60, alpha=0.9)
        plt.colorbar(sc, ax=ax, label='Depth Z (m)')
        ax.set_title(f'O1 — sparse LiDAR pointmap  [{CAMERA}]')
    elif POINTMAP_MODE == 'o2_moge_affine':
        im = ax.imshow(_Z, cmap='plasma', vmin=0, vmax=60)
        plt.colorbar(im, ax=ax, label='Metric depth Z (m)')
        ax.scatter(_u_prim, _v_prim, c='white', s=1, alpha=0.5)
        ax.set_title(f'O2 — MoGe + global affine  (Z = {_primary_viz["a"]:.3f}×Z_moge + {_primary_viz["b"]:.3f})  [{CAMERA}]')
    elif POINTMAP_MODE == 'o3_local_affine':
        im = ax.imshow(_Z, cmap='plasma')
        plt.colorbar(im, ax=ax, label='Relative depth (MoGe units)')
        ax.scatter(_u_prim, _v_prim, c='white', s=1, alpha=0.5)
        ax.set_title(f'O3 — MoGe base map (per-object calibration applied per-inference)  [{CAMERA}]')
    elif POINTMAP_MODE == 'o4_dense_pc':
        im = ax.imshow(_Z, cmap='plasma', vmin=0, vmax=60)
        plt.colorbar(im, ax=ax, label='Metric depth (m)')
        ax.scatter(_u_prim, _v_prim, c='white', s=1, alpha=0.5)
        ax.set_title(f'O4 — CompletionFormer dense metric depth  [{CAMERA}]')
    else:
        im = ax.imshow(_Z, cmap='plasma')
        plt.colorbar(im, ax=ax, label='Relative depth (MoGe units)')
        ax.set_title(f'Baseline — MoGe relative depth  [{CAMERA}]')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
# Quick 2D projection of meshes onto image
def project(pts, K):
    """(N,3) camera-space pts -> (N,2) pixel coords. Returns -9999 for behind-camera pts."""
    z = pts[:, 2]
    valid = z > 0
    uv = np.full((len(pts), 2), -9999.0, dtype=np.float32)
    uv[valid, 0] = pts[valid, 0] / z[valid] * K[0, 0] + K[0, 2]
    uv[valid, 1] = pts[valid, 1] / z[valid] * K[1, 1] + K[1, 2]
    return uv


obj_colors = [(80, 220, 80), (80, 80, 255), (255, 200, 0), (200, 0, 255), (0, 220, 220)]

n_cams = len(CAMERAS)
_ncols = min(n_cams, 3)
_nrows = (n_cams + _ncols - 1) // _ncols
fig, axes = plt.subplots(_nrows, _ncols, figsize=(14, 5 * _nrows), squeeze=False)
axes_flat = [ax for row in axes for ax in row]

for _i, _cam in enumerate(CAMERAS):
    ax   = axes_flat[_i]
    _img = cam_data[_cam]['img_rgb']
    _K   = cam_data[_cam]['K']
    _H, _W = _img.shape[:2]
    vis  = _img.copy()

    for r in body_results_all[_cam]:
        uv = project(r['vertices'], _K)
        ok = (uv[:, 0] >= 0) & (uv[:, 0] < _W) & (uv[:, 1] >= 0) & (uv[:, 1] < _H)
        for u, v in uv[ok].astype(int):
            cv2.circle(vis, (u, v), 1, (255, 80, 80), -1)

    for i, r in enumerate(obj_results_all[_cam]):
        uv  = project(r['vertices'], _K)
        ok  = (uv[:, 0] >= 0) & (uv[:, 0] < _W) & (uv[:, 1] >= 0) & (uv[:, 1] < _H)
        col = obj_colors[i % len(obj_colors)]
        for u, v in uv[ok].astype(int):
            cv2.circle(vis, (u, v), 1, col, -1)

    n_b = len(body_results_all[_cam])
    n_o = len(obj_results_all[_cam])
    ax.imshow(vis)
    ax.axis('off')
    ax.set_title(f'{_cam}: {n_b} ped [red], {n_o} obj [colour]', fontsize=9)

for ax in axes_flat[n_cams:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## 8. Module: Orientation Estimation

- **Pedestrians**: `facing_direction()` — shoulder line → nose-disambiguated unit vector in XZ plane. Fully resolved.
- **Objects**: `None` — 180° yaw ambiguity is accepted for stage 1. PCA long axis only.

The orientation result is a unit vector in camera XZ space for pedestrians, `None` for objects.

In [ ]:
# ── Orientation estimation ────────────────────────────────────────────────────

_NOSE       = 0
_L_SHOULDER = 5
_R_SHOULDER = 6


def facing_direction(joints):
    """
    Estimate horizontal facing direction of a pedestrian from MHR70 skeleton.

    Parameters
    ----------
    joints : (70, 3) float32  MHR70 keypoints in body-relative space

    Returns
    -------
    fwd : (3,) float32  unit vector in XZ plane, Y=0. Fully resolved (no 180° ambiguity).
    """
    shoulder_vec = joints[_R_SHOULDER] - joints[_L_SHOULDER]
    sv_xz = np.array([shoulder_vec[0], 0.0, shoulder_vec[2]], dtype=np.float32)
    sv_xz /= np.linalg.norm(sv_xz) + 1e-8

    # 90° rotation in XZ gives a facing candidate
    fwd = np.array([-sv_xz[2], 0.0, sv_xz[0]], dtype=np.float32)

    # Disambiguate via nose: nose should be in front of the shoulder midpoint
    shoulder_mid = (joints[_L_SHOULDER] + joints[_R_SHOULDER]) / 2.0
    if np.dot(fwd, joints[_NOSE] - shoulder_mid) < 0:
        fwd = -fwd

    return fwd


def estimate_orientation(result, kind):
    """
    Estimate orientation for a single detection.

    Parameters
    ----------
    result : dict from body_results or obj_results
    kind   : 'pedestrian' | 'object'

    Returns
    -------
    fwd : (3,) unit vector in camera XZ plane, or None (ambiguous)
    """
    if kind == 'pedestrian':
        return facing_direction(result['joints_3d'])
    # Objects: ambiguous for stage 1
    return None


# Attach orientation to all cameras
for _cam in CAMERAS:
    for r in body_results_all[_cam]:
        r['orientation_fwd'] = estimate_orientation(r, 'pedestrian')
    for r in obj_results_all[_cam]:
        r['orientation_fwd'] = estimate_orientation(r, 'object')

# Primary camera aliases
body_results = body_results_all[CAMERA]
obj_results  = obj_results_all[CAMERA]

print('Orientation estimation done.')
for i, r in enumerate(body_results):
    print(f'  Pedestrian {i}: fwd = {r["orientation_fwd"].round(3)}')
for i, r in enumerate(obj_results):
    print(f'  Object {i} ({r["prompt"]}): orientation = ambiguous (stage 1)')

## 9. Module: OBB Fitting

- **Pedestrians**: `compute_obb_pedestrian()` — projects mesh vertices onto `(right, up, fwd)` axes derived from `facing_direction`. Stays in camera (body-relative) space.
- **Objects**: `compute_obb_gravity_aligned()` — PCA footprint in ego frame. No ground anchoring (stage 1). 180° yaw ambiguity in the long axis.

In [ ]:
# ── OBB Fitting ───────────────────────────────────────────────────────────────

_BBOX_EDGES = [
    (0,1),(1,2),(2,3),(3,0),   # bottom face
    (4,5),(5,6),(6,7),(7,4),   # top face
    (0,4),(1,5),(2,6),(3,7),   # verticals
]


def compute_obb_pedestrian(vertices, fwd):
    """
    Fit an OBB aligned to the pedestrian's facing direction.

    Parameters
    ----------
    vertices : (V, 3) float32  mesh vertices in camera space (R3)
    fwd      : (3,)   float32  unit facing vector in XZ plane (Y=0)

    Returns
    -------
    corners : (8, 3) float32  OBB corners in camera space
    center  : (3,)   float32
    dims    : (3,)   float32  [width, height, depth]
    """
    right = np.array([fwd[2], 0.0, -fwd[0]], dtype=np.float32)   # 90° CW from fwd in XZ
    up    = np.array([0.0, 1.0,  0.0], dtype=np.float32)

    r_proj = vertices @ right
    y_proj = vertices[:, 1]
    f_proj = vertices @ fwd

    r_min, r_max = float(r_proj.min()), float(r_proj.max())
    y_min, y_max = float(y_proj.min()), float(y_proj.max())
    f_min, f_max = float(f_proj.min()), float(f_proj.max())

    corners = np.array([
        r_min*right + y_min*up + f_min*fwd,
        r_max*right + y_min*up + f_min*fwd,
        r_max*right + y_max*up + f_min*fwd,
        r_min*right + y_max*up + f_min*fwd,
        r_min*right + y_min*up + f_max*fwd,
        r_max*right + y_min*up + f_max*fwd,
        r_max*right + y_max*up + f_max*fwd,
        r_min*right + y_max*up + f_max*fwd,
    ], dtype=np.float32)

    center = corners.mean(axis=0)
    dims   = np.array([r_max-r_min, y_max-y_min, f_max-f_min], dtype=np.float32)
    return corners, center, dims


def compute_obb_gravity_aligned(verts_r3, R_c2e, t_c2e, ground_z=None,
                                 height_pct=(1, 99), height_clamp=(0.9, 2.8)):
    """
    Fit a gravity-aligned OBB to mesh vertices, in ego frame.

    Footprint (yaw / length / width) from PCA of mesh XY footprint in ego space.
    Yaw has 180° ambiguity (no LiDAR to resolve it at stage 1).

    Parameters
    ----------
    verts_r3     : (V, 3) float32  mesh vertices in R3 camera space
    R_c2e        : (3, 3) float64  rotation camera -> ego
    t_c2e        : (3,)   float64  translation camera -> ego
    ground_z     : float | None    ego-frame ground height (None = image-only stage)
    height_pct   : (low, high)     percentiles for robust height estimate
    height_clamp : (min_h, max_h)  plausible height range [m]

    Returns
    -------
    corners_ego : (8, 3) float64  OBB corners in ego frame
    center_ego  : (3,)   float64
    dims        : (3,)   float64  [length, width, height]
    yaw_rad     : float           yaw angle of long axis (+-pi ambiguous)
    """
    verts_ego = (R_c2e @ verts_r3.T).T + t_c2e

    # PCA on XY footprint
    xy    = verts_ego[:, :2]
    xy_c  = xy - xy.mean(axis=0)
    _, evecs = np.linalg.eigh(np.cov(xy_c.T))
    evecs = evecs[:, ::-1]   # largest eigenvalue first

    ax_l = np.array([evecs[0,0], evecs[1,0], 0.], dtype=np.float64)   # long axis
    ax_w = np.array([evecs[0,1], evecs[1,1], 0.], dtype=np.float64)   # short axis
    ax_h = np.array([0., 0., 1.],                 dtype=np.float64)   # height (ego Z-up)

    # Ensure right-hand system
    if ax_l[0]*ax_w[1] - ax_l[1]*ax_w[0] < 0:
        ax_w = -ax_w
    axes  = np.stack([ax_l, ax_w, ax_h], axis=1)  # (3, 3)

    proj  = verts_ego @ axes
    min_p = proj.min(axis=0).copy()
    max_p = proj.max(axis=0).copy()

    if ground_z is not None:
        vz          = verts_ego[:, 2]
        raw_h       = float(np.percentile(vz, height_pct[1]) - np.percentile(vz, height_pct[0]))
        robust_h    = float(np.clip(raw_h, height_clamp[0], height_clamp[1]))
        min_p[2]    = ground_z
        max_p[2]    = ground_z + robust_h

    mid_p      = (min_p + max_p) / 2.0
    center_ego = axes @ mid_p

    offsets = np.array([
        [min_p[0]-mid_p[0], min_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], min_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], max_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], max_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], min_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], min_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], max_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], max_p[1]-mid_p[1], max_p[2]-mid_p[2]],
    ])
    corners_ego = (axes @ offsets.T).T + center_ego
    dims        = max_p - min_p
    yaw_rad     = float(np.arctan2(evecs[1,0], evecs[0,0]))

    return corners_ego, center_ego, dims, yaw_rad


def fit_obb(result, kind, R_c2e, t_c2e):
    """
    Fit an OBB for a single detection.

    Parameters
    ----------
    result : dict
    kind   : 'pedestrian' | 'object'
    R_c2e  : (3, 3)  camera -> ego rotation
    t_c2e  : (3,)    camera -> ego translation

    Returns
    -------
    Adds 'obb_corners', 'obb_center', 'obb_dims' (and 'obb_yaw' for objects) to result in-place.
    """
    verts = result['vertices']
    if kind == 'pedestrian':
        fwd = result['orientation_fwd']
        corners, center, dims = compute_obb_pedestrian(verts, fwd)
        result['obb_corners'] = corners    # camera space
        result['obb_center']  = center
        result['obb_dims']    = dims
        result['obb_yaw']     = float(np.arctan2(fwd[0], fwd[2]))  # approx yaw in camera
    else:
        corners, center, dims, yaw = compute_obb_gravity_aligned(
            verts, R_c2e, t_c2e, ground_z=None
        )
        result['obb_corners'] = corners    # ego space
        result['obb_center']  = center
        result['obb_dims']    = dims
        result['obb_yaw']     = yaw        # +-pi ambiguous


# Fit OBBs for all cameras
for _cam in CAMERAS:
    _R_c2e = cam_data[_cam]['R_c2e']
    _t_c2e = cam_data[_cam]['t_c2e']
    for r in body_results_all[_cam]:
        fit_obb(r, 'pedestrian', _R_c2e, _t_c2e)
    for r in obj_results_all[_cam]:
        fit_obb(r, 'object', _R_c2e, _t_c2e)

# Primary camera aliases
body_results = body_results_all[CAMERA]
obj_results  = obj_results_all[CAMERA]

print('OBB fitting done.')
for _cam in CAMERAS:
    _nb = len(body_results_all[_cam])
    _no = len(obj_results_all[_cam])
    print(f'  [{_cam}] {_nb} body, {_no} objects')
    for i, r in enumerate(body_results_all[_cam]):
        print(f'    Pedestrian {i}: dims (W×H×D) = {r["obb_dims"].round(3)} m (camera-relative)')
    for i, r in enumerate(obj_results_all[_cam]):
        print(f'    Object {i} ({r["prompt"]}): dims (L×W×H) = {r["obb_dims"].round(3)} m (ego)')

In [ ]:
# ── OBB size filter — discard definitely-too-small pseudo labels ──────────────


def obb_dims_valid(result, kind, min_dims_cfg):
    """
    Return True if the OBB meets per-class minimum dimension requirements.

    Pedestrian dims are camera-space [W, H, D]; H (index 1) checked.
    Object dims are ego-space [L, W, H];        L (index 0) and H (index 2) checked.
    """
    key = 'pedestrian' if kind == 'pedestrian' else result['prompt']
    cfg = min_dims_cfg.get(key, {})
    if not cfg:
        return True
    dims = result['obb_dims']
    if kind == 'pedestrian':
        if cfg.get('min_h') is not None and dims[1] < cfg['min_h']:
            return False
    else:
        if cfg.get('min_l') is not None and dims[0] < cfg['min_l']:
            return False
        if cfg.get('min_h') is not None and dims[2] < cfg['min_h']:
            return False
    return True


if USE_OBB_MIN_DIMS_FILTER:
    total_dropped_body, total_dropped_obj = 0, 0
    for _cam in CAMERAS:
        _n_body = len(body_results_all[_cam])
        _n_obj  = len(obj_results_all[_cam])
        body_results_all[_cam] = [
            r for r in body_results_all[_cam] if obb_dims_valid(r, 'pedestrian', OBB_MIN_DIMS)
        ]
        obj_results_all[_cam] = [
            r for r in obj_results_all[_cam]  if obb_dims_valid(r, 'object',     OBB_MIN_DIMS)
        ]
        _d_body = _n_body - len(body_results_all[_cam])
        _d_obj  = _n_obj  - len(obj_results_all[_cam])
        total_dropped_body += _d_body
        total_dropped_obj  += _d_obj
        if _d_body or _d_obj:
            print(f'  [{_cam}] dropped {_d_body} body, {_d_obj} object (too small)')
    
    # Primary camera aliases
    body_results = body_results_all[CAMERA]
    obj_results  = obj_results_all[CAMERA]
    
    print(f'OBB size filter done — dropped {total_dropped_body} body, {total_dropped_obj} object OBB(s) total.')
    print(f'Remaining ({CAMERA}): {len(body_results)} body, {len(obj_results)} objects')
else:
    body_results = body_results_all[CAMERA]
    obj_results  = obj_results_all[CAMERA]
    print('OBB size filter: disabled.')

In [ ]:
# ── Cyclist / motorcyclist OBB merge ─────────────────────────────────────────
# When a pedestrian body result (SAM3D Body) and a bicycle or motorcycle result
# (SAM3D Objects) originate from the same physical person-on-vehicle, their
# OBBs are merged:
#   1. The body OBB center (camera space) is transformed to ego space.
#   2. If the distance to a bicycle/motorcycle OBB center (ego space) is < RIDER_MERGE_DIST,
#      vertices from both meshes are combined and a fresh OBB is recomputed.
#   3. The vehicle class label is kept; the pedestrian body entry is removed.
#
# Distance rationale: body OBB center (torso, ~1 m height) vs. bicycle OBB center
# (frame, ~0.5 m height) are typically 0.6-1.0 m apart in 3D ego space even when
# the rider is directly mounted. 1.5 m is tight enough to avoid accidentally
# merging a pedestrian walking next to a parked bike.

RIDER_MERGE_DIST = 1.5   # metres (3D ego space)
RIDER_CLASSES    = {'bicycle', 'motorcycle'}


def merge_rider_obbs(body_res, obj_res, R_c2e, t_c2e, dist_thresh=RIDER_MERGE_DIST):
    """
    Merge pedestrian body OBBs with co-located bicycle / motorcycle OBBs.
    Returns (new_body_res, obj_res) — matched body entries are removed.
    """
    body_used = set()
    for bi, br in enumerate(body_res):
        # body obb_center is in camera space -> transform to ego
        bc_ego = R_c2e @ br['obb_center'].astype(np.float64) + t_c2e

        best_oi, best_d = None, float('inf')
        for oi, or_ in enumerate(obj_res):
            if or_['prompt'] not in RIDER_CLASSES:
                continue
            d = float(np.linalg.norm(bc_ego - np.array(or_['obb_center'], dtype=np.float64)))
            if d <= dist_thresh and d < best_d:
                best_d, best_oi = d, oi

        if best_oi is None:
            continue

        or_      = obj_res[best_oi]
        combined = np.concatenate([br['vertices'], or_['vertices']], axis=0).astype(np.float32)
        corners, center, dims, yaw = compute_obb_gravity_aligned(
            combined, R_c2e, t_c2e, ground_z=None
        )
        # OBB is fitted to combined vertices; individual meshes are kept separate
        or_['rider_vertices'] = br['vertices']
        or_['rider_faces']    = br['faces']
        or_['obb_corners'] = corners
        or_['obb_center']  = center
        or_['obb_dims']    = dims
        or_['obb_yaw']     = yaw
        body_used.add(bi)
        print(f'  [rider merge] pedestrian -> "{or_["prompt"]}"  (d={best_d:.2f} m)')

    return [br for bi, br in enumerate(body_res) if bi not in body_used], obj_res


# Merge riders for all cameras
for _cam in CAMERAS:
    _R_c2e = cam_data[_cam]['R_c2e']
    _t_c2e = cam_data[_cam]['t_c2e']
    _n_body_before = len(body_results_all[_cam])
    body_results_all[_cam], obj_results_all[_cam] = merge_rider_obbs(
        body_results_all[_cam], obj_results_all[_cam], _R_c2e, _t_c2e
    )
    _n_merged = _n_body_before - len(body_results_all[_cam])
    print(f'  [{_cam}] Rider merge: {_n_merged} pedestrian(s) merged. Body: {_n_body_before} -> {len(body_results_all[_cam])}')

# Primary camera aliases
body_results = body_results_all[CAMERA]
obj_results  = obj_results_all[CAMERA]

_total_b = sum(len(body_results_all[c]) for c in CAMERAS)
_total_o = sum(len(obj_results_all[c])  for c in CAMERAS)
print(f'\nTotal across all cameras: {_total_b} body, {_total_o} objects')
print(f'Primary camera ({CAMERA}): {len(body_results)} body, {len(obj_results)} objects')

## 10. 2D OBB Visualization
Project OBB wireframes onto the image.

In [ ]:
def draw_obb_on_image(img_rgb, corners, K, color, R_c2e=None, t_c2e=None, in_ego=False):
    """
    Draw an OBB wireframe on an image by projecting 8 corners.

    Parameters
    ----------
    corners   : (8, 3)  OBB corners — camera space OR ego space
    in_ego    : bool    if True, transform corners ego -> camera first
    """
    vis = img_rgb.copy()
    _H, _W = vis.shape[:2]
    c3d = corners.astype(np.float64)
    if in_ego and R_c2e is not None:
        # ego -> camera: R_c2e^T @ (p - t_c2e)
        c3d = (R_c2e.T @ (c3d - t_c2e).T).T
    uv = project(c3d.astype(np.float32), K)
    for e0, e1 in _BBOX_EDGES:
        p0, p1 = uv[e0], uv[e1]
        if (p0 == -9999).any() or (p1 == -9999).any():
            continue
        pt0 = (int(np.clip(p0[0], 0, _W-1)), int(np.clip(p0[1], 0, _H-1)))
        pt1 = (int(np.clip(p1[0], 0, _W-1)), int(np.clip(p1[1], 0, _H-1)))
        cv2.line(vis, pt0, pt1, color, 2)
    return vis


n_cams = len(CAMERAS)
_ncols = min(n_cams, 3)
_nrows = (n_cams + _ncols - 1) // _ncols
fig, axes = plt.subplots(_nrows, _ncols, figsize=(14, 5 * _nrows), squeeze=False)
axes_flat = [ax for row in axes for ax in row]

for _i, _cam in enumerate(CAMERAS):
    ax     = axes_flat[_i]
    _img   = cam_data[_cam]['img_rgb']
    _K     = cam_data[_cam]['K']
    _R_c2e = cam_data[_cam]['R_c2e']
    _t_c2e = cam_data[_cam]['t_c2e']
    vis    = _img.copy()
    for r in body_results_all[_cam]:
        vis = draw_obb_on_image(vis, r['obb_corners'], _K, color=(255, 60, 60), in_ego=False)
    for r in obj_results_all[_cam]:
        vis = draw_obb_on_image(vis, r['obb_corners'], _K, color=(60, 220, 60),
                                R_c2e=_R_c2e, t_c2e=_t_c2e, in_ego=True)
    ax.imshow(vis)
    ax.axis('off')
    ax.set_title(f'{_cam}', fontsize=9)

for ax in axes_flat[n_cams:]:
    ax.axis('off')

plt.suptitle('OBBs — pedestrians [red, cam space] | objects [green, ego space]', y=1.01)
plt.tight_layout()
plt.show()

## 11. 3D Visualization (plotly)
Interactive scene with meshes and OBBs in ego frame. All pedestrian geometry is first converted from camera to ego space.

In [25]:
def cam_to_ego(pts, R_c2e, t_c2e):
    """Transform (N,3) points from camera (R3) to ego frame."""
    return (R_c2e @ pts.T).T + t_c2e


OBJ_COLORS = [
    'rgb(64,221,64)', 'rgb(64,64,255)', 'rgb(255,200,0)',
    'rgb(200,0,255)', 'rgb(0,221,221)', 'rgb(255,100,0)',
]

traces = []


def _obb_lines(corners, color):
    """Scatter3d trace for one OBB wireframe (12 edges, None-separated)."""
    xs, ys, zs = [], [], []
    for e0, e1 in _BBOX_EDGES:
        xs += [corners[e0, 0], corners[e1, 0], None]
        ys += [corners[e0, 1], corners[e1, 1], None]
        zs += [corners[e0, 2], corners[e1, 2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                        line=dict(color=color, width=3), showlegend=False)


# # ── Aggregate all cameras into ego frame ──────────────────────────────────────
# _ped_counter, _obj_counter = 0, 0
# for _cam in CAMERAS:
#     _R = cam_data[_cam]['R_c2e']
#     _t = cam_data[_cam]['t_c2e']

#     # Pedestrian meshes (red)
#     for r in body_results_all[_cam]:
#         verts_ego = cam_to_ego(r['vertices'].astype(np.float64), _R, _t)
#         f = r['faces']
#         traces.append(go.Mesh3d(
#             x=verts_ego[:, 0], y=verts_ego[:, 1], z=verts_ego[:, 2],
#             i=f[:, 0], j=f[:, 1], k=f[:, 2],
#             color='rgb(255,64,64)', opacity=0.7,
#             name=f'pedestrian_{_ped_counter} ({_cam})', showlegend=True,
#         ))
#         corners_ego = cam_to_ego(r['obb_corners'].astype(np.float64), _R, _t)
#         traces.append(_obb_lines(corners_ego, 'rgb(170,0,0)'))
#         center_ego = cam_to_ego(r['obb_center'].reshape(1, 3).astype(np.float64), _R, _t)[0]
#         fwd_ego    = _R @ r['orientation_fwd'].astype(np.float64)
#         tip_ego    = center_ego + fwd_ego * 0.5
#         traces.append(go.Scatter3d(
#             x=[center_ego[0], tip_ego[0]],
#             y=[center_ego[1], tip_ego[1]],
#             z=[center_ego[2], tip_ego[2]],
#             mode='lines', line=dict(color='rgb(255,255,0)', width=6),
#             showlegend=False,
#         ))
#         _ped_counter += 1

#     # Object meshes + OBBs (coloured)
#     for r in obj_results_all[_cam]:
#         col       = OBJ_COLORS[_obj_counter % len(OBJ_COLORS)]
#         verts_ego = cam_to_ego(r['vertices'].astype(np.float64), _R, _t)
#         f         = r['faces']
#         traces.append(go.Mesh3d(
#             x=verts_ego[:, 0], y=verts_ego[:, 1], z=verts_ego[:, 2],
#             i=f[:, 0], j=f[:, 1], k=f[:, 2],
#             color=col, opacity=0.7,
#             name=f'{r["prompt"]}_{_obj_counter} ({_cam})', showlegend=True,
#         ))
#         if 'rider_vertices' in r:
#             rv = cam_to_ego(r['rider_vertices'].astype(np.float64), _R, _t)
#             rf = r['rider_faces']
#             traces.append(go.Mesh3d(
#                 x=rv[:, 0], y=rv[:, 1], z=rv[:, 2],
#                 i=rf[:, 0], j=rf[:, 1], k=rf[:, 2],
#                 color=col, opacity=0.7, showlegend=False,
#             ))
#         traces.append(_obb_lines(r['obb_corners'].astype(np.float64), col))
#         _obj_counter += 1

# fig = go.Figure(data=traces)
# fig.update_layout(
#     title=f'Auto-labeling Stage 1 — ego frame | scene {SCENE_IDX} frame {FRAME_IDX}',
#     scene=dict(
#         xaxis_title='X ego', yaxis_title='Y ego', zaxis_title='Z ego',
#         aspectmode='data',
#     ),
#     margin=dict(l=0, r=0, b=0, t=40),
# )
# fig.show()

## 12. BEV Visualization — LiDAR + Pseudo Labels + Ground Truth
Interactive bird's-eye-view plot. LiDAR coloured by height, pseudo-label OBBs as solid outlines, GT annotations as white dashed boxes.

> **Note:** GT annotations exist only on keyframes. If the selected frame is not a keyframe the nearest preceding keyframe's GT is used.

In [ ]:
# cam_sd, pts_ego, lidar_token are already loaded in Section 3.5.

# ── GT annotations ────────────────────────────────────────────────────────────
# GT exists only on keyframes — walk back through sample_data to the nearest one.
gt_boxes_ego = []
_token = cam_sd_token
while _token:
    _sd = nusc.get('sample_data', _token)
    if _sd['is_key_frame']:
        _sample_gt      = nusc.get('sample', _sd['sample_token'])
        for ann_token in _sample_gt['anns']:
            ann        = nusc.get('sample_annotation', ann_token)
            center_g   = np.array(ann['translation'], dtype=np.float64)
            half_w     = ann['size'][0] / 2.0   # size = [width, length, height]
            half_l     = ann['size'][1] / 2.0
            # Global → anchor ego frame (same frame as pts_ego)
            center_e   = R_e2g_anchor.T @ (center_g - t_e2g_anchor)
            # Yaw in anchor ego frame
            R_box_e    = R_e2g_anchor.T @ Quaternion(ann['rotation']).rotation_matrix
            yaw        = float(np.arctan2(R_box_e[1, 0], R_box_e[0, 0]))
            cy, sy     = np.cos(yaw), np.sin(yaw)
            corners_2d = center_e[:2] + np.array([
                [+half_l*cy - half_w*sy, +half_l*sy + half_w*cy],
                [+half_l*cy + half_w*sy, +half_l*sy - half_w*cy],
                [-half_l*cy + half_w*sy, -half_l*sy - half_w*cy],
                [-half_l*cy - half_w*sy, -half_l*sy + half_w*cy],
            ])
            gt_boxes_ego.append({
                'corners_2d': corners_2d,
                'category':   ann['category_name'],
            })
        print(f'GT: {len(gt_boxes_ego)} annotation(s) from keyframe {_sd["sample_token"][:8]}...')
        break
    _token = _sd['prev'] if _sd['prev'] else ''

if not gt_boxes_ego and not _token:
    print('GT: no preceding keyframe found.')

# ── BEV plot ──────────────────────────────────────────────────────────────────
_rng  = np.random.default_rng(0)
_idx  = _rng.choice(len(pts_ego), min(40_000, len(pts_ego)), replace=False)
pts_d = pts_ego[_idx]
_mask = (np.abs(pts_d[:, 0]) < 60) & (np.abs(pts_d[:, 1]) < 30)
pts_d = pts_d[_mask]

bev_traces = []

# LiDAR points coloured by height
bev_traces.append(go.Scatter(
    x=pts_d[:, 0], y=pts_d[:, 1],
    mode='markers',
    marker=dict(size=1.5, color=pts_d[:, 2], colorscale='Viridis',
                cmin=-2, cmax=3, showscale=True,
                colorbar=dict(title='Height (m)', thickness=12, x=1.01)),
    name='LiDAR', hoverinfo='skip',
))

# GT boxes — white dashed, grouped by top-level category
_GT_COLORS = {
    'vehicle':        'rgb(255,255,255)',
    'human':          'rgb(255,210,100)',
    'movable_object': 'rgb(180,180,180)',
    'static_object':  'rgb(140,140,140)',
}
_shown_gt = set()
for b in gt_boxes_ego:
    top_cat = b['category'].split('.')[0]
    color   = _GT_COLORS.get(top_cat, 'rgb(200,200,200)')
    name    = f'GT {top_cat}'
    c       = b['corners_2d']
    bev_traces.append(go.Scatter(
        x=[c[0,0], c[1,0], c[2,0], c[3,0], c[0,0]],
        y=[c[0,1], c[1,1], c[2,1], c[3,1], c[0,1]],
        mode='lines',
        line=dict(color=color, width=2, dash='dash'),
        name=name, legendgroup=name,
        showlegend=(name not in _shown_gt),
    ))
    _shown_gt.add(name)

# Pseudo-label OBBs — solid, coloured by class
_PSEUDO_COLORS = {
    'pedestrian':           'rgb(255,80,80)',
    'car':                  'rgb(80,220,80)',
    'bicycle and cyclist':  'rgb(80,120,255)',
    'motorcycle':           'rgb(255,160,0)',
    'bus':                  'rgb(200,80,255)',
    'truck':                'rgb(0,210,210)',
    'construction vehicle': 'rgb(255,80,180)',
    'trailer':              'rgb(160,220,60)',
}

def _footprint_xy(corners_xy):
    """Sort corners by angle from centroid to get a correct closed polygon."""
    cx, cy  = corners_xy.mean(axis=0)
    angles  = np.arctan2(corners_xy[:, 1] - cy, corners_xy[:, 0] - cx)
    return corners_xy[np.argsort(angles)]

# Aggregate pseudo-label OBBs from all cameras
_shown_pseudo = set()
for _cam in CAMERAS:
    _R = cam_data[_cam]['R_c2e']
    _t = cam_data[_cam]['t_c2e']

    for r in body_results_all[_cam]:
        corners_e = cam_to_ego(r['obb_corners'].astype(np.float64), _R, _t)
        foot      = _footprint_xy(corners_e[:, :2])
        color     = _PSEUDO_COLORS['pedestrian']
        name      = 'pseudo pedestrian'
        bev_traces.append(go.Scatter(
            x=list(foot[:, 0]) + [foot[0, 0]],
            y=list(foot[:, 1]) + [foot[0, 1]],
            mode='lines', line=dict(color=color, width=2),
            name=name, legendgroup=name,
            showlegend=(name not in _shown_pseudo),
        ))
        _shown_pseudo.add(name)

    for r in obj_results_all[_cam]:
        if BEV_DROP_NON_LOCAL and r.get('ptmap_mode', 'local') != 'local':
            continue
        foot  = _footprint_xy(r['obb_corners'].astype(np.float64)[:, :2])
        color = _PSEUDO_COLORS.get(r['prompt'], 'rgb(200,200,200)')
        name  = f'pseudo {r["prompt"]}'
        bev_traces.append(go.Scatter(
            x=list(foot[:, 0]) + [foot[0, 0]],
            y=list(foot[:, 1]) + [foot[0, 1]],
            mode='lines', line=dict(color=color, width=2),
            name=name, legendgroup=name,
            showlegend=(name not in _shown_pseudo),
        ))
        _shown_pseudo.add(name)

# Ego vehicle marker
bev_traces.append(go.Scatter(
    x=[0], y=[0], mode='markers',
    marker=dict(symbol='arrow-up', angle=90, size=16,
                color='yellow', line=dict(color='black', width=1)),
    name='ego vehicle', showlegend=True,
))

fig_bev = go.Figure(data=bev_traces)
fig_bev.update_layout(
    title=(f'BEV — Scene {SCENE_IDX}, frame {FRAME_IDX} | '
           f'solid = pseudo labels | dashed = GT'),
    xaxis=dict(title='X ego (forward, m)', range=[-10, 60],
               scaleanchor='y', scaleratio=1,
               showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False),
    yaxis=dict(title='Y ego (left, m)', range=[-30, 30],
               showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False),
    plot_bgcolor='rgb(15,15,25)',
    paper_bgcolor='rgb(15,15,25)',
    font=dict(color='white'),
    legend=dict(bgcolor='rgba(0,0,0,0.5)', font=dict(size=11)),
    width=1000, height=580,
    margin=dict(l=0, r=80, t=40, b=0),
)
fig_bev.show()

## 12. Summary
Print a quick summary table of all detections.

In [ ]:
print(f'Scene {SCENE_IDX}, frame {FRAME_IDX} — multi-camera summary')
print()
for _cam in CAMERAS:
    _body = body_results_all[_cam]
    _objs = obj_results_all[_cam]
    print(f'  {_cam}: {len(_body)} pedestrian(s), {len(_objs)} object(s)')

print(f'\nPrimary camera ({CAMERA}) detail:')
print(f'{"Class":<25} {"Dims (m)":>30} {"Yaw (rad)":>12} {"Score":>8}')
print('-' * 80)

for i, r in enumerate(body_results):
    d = r['obb_dims']
    print(f'{"pedestrian_" + str(i):<25} W={d[0]:.2f} H={d[1]:.2f} D={d[2]:.2f} (camera) '           f'{r["obb_yaw"]:>10.3f}  {"—":>8}')

for i, r in enumerate(obj_results):
    d     = r['obb_dims']
    label = r['prompt'] + '_' + str(i)
    print(f'{label:<25} L={d[0]:.2f} W={d[1]:.2f} H={d[2]:.2f} (ego)   '           f'{r["obb_yaw"]:>10.3f}* {r["score"]:>8.3f}')

print('\n* yaw has 180° ambiguity for objects at stage 1 (no LiDAR)')